<a href="https://colab.research.google.com/github/shinejeweljaipur/-fifa-eda-project/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =============================================================
# FIFA WORLD CUP 2026 - PLAYER PERFORMANCE : COMPLETE EDA SCRIPT
# Covers almost every common NumPy + Pandas operation used in EDA
# =============================================================

import numpy as np
import pandas as pd

# -------------------------------------------------------------
# STEP 0 : LOAD RAW DATA
# -------------------------------------------------------------
RAW_PATH = 'fifa_world_cup_2026_player_performance.csv'   # apna path yahan set karo
df_raw = pd.read_csv(RAW_PATH)
df = df_raw.copy()          # working copy — raw stays untouched

# -------------------------------------------------------------
# STEP 1 : BASIC STRUCTURE / OVERVIEW
# -------------------------------------------------------------
print("Shape (rows, cols):", df.shape)
print("\nColumn names:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:\n", df.head())
print("\nLast 5 rows:\n", df.tail())
print("\nRandom sample:\n", df.sample(5))
print("\nFull info:")
df.info()

# Memory usage
print("\nMemory usage (MB):", df.memory_usage(deep=True).sum() / 1e6)

# -------------------------------------------------------------
# STEP 2 : STATISTICAL SUMMARY
# -------------------------------------------------------------
print("\nNumeric summary:\n", df.describe())
print("\nCategorical summary:\n", df.describe(include='object'))
print("\nAll columns summary:\n", df.describe(include='all'))

# Individual stats (NumPy under the hood)
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print("\nNumeric columns:", num_cols)

print("\nMean of goals:", np.mean(df['goals']))
print("Median of goals:", np.median(df['goals']))
print("Std of goals:", np.std(df['goals']))
print("Variance of goals:", np.var(df['goals']))
print("Min/Max of age:", np.min(df['age']), np.max(df['age']))
print("25th/50th/75th percentile of market_value_eur:",
      np.percentile(df['market_value_eur'], [25, 50, 75]))
print("Sum of total goals scored:", np.sum(df['goals']))

# -------------------------------------------------------------
# STEP 3 : MISSING VALUES
# -------------------------------------------------------------
print("\nMissing values per column:\n", df.isnull().sum())
print("\n% missing per column:\n", (df.isnull().sum() / len(df)) * 100)
print("\nTotal missing cells:", df.isnull().sum().sum())
print("Rows with at least one missing value:", df.isnull().any(axis=1).sum())

# Visual-style missing check (heatmap needs matplotlib/seaborn — optional)
# import seaborn as sns
# import matplotlib.pyplot as plt
# sns.heatmap(df.isnull(), cbar=False)
# plt.show()

# Handling missing values (choose approach based on column)
df_filled_mean   = df.fillna(df.mean(numeric_only=True))          # numeric -> mean
df_dropped       = df.dropna()                                    # drop rows with NA
df_ffill         = df.ffill()                                      # forward fill
df_bfill         = df.bfill()                                      # backward fill

# -------------------------------------------------------------
# STEP 4 : DUPLICATES
# -------------------------------------------------------------
print("\nDuplicate rows:", df.duplicated().sum())
df_no_dupes = df.drop_duplicates()

# -------------------------------------------------------------
# STEP 5 : UNIQUE VALUES / VALUE COUNTS (categorical exploration)
# -------------------------------------------------------------
print("\nUnique teams:", df['team'].nunique())
print("Unique positions:", df['position'].unique())
print("\nValue counts - position:\n", df['position'].value_counts())
print("\nValue counts - nationality (top 10):\n", df['nationality'].value_counts().head(10))
print("\nValue counts - preferred_foot:\n", df['preferred_foot'].value_counts(normalize=True) * 100)

# -------------------------------------------------------------
# STEP 6 : FILTERING / INDEXING
# -------------------------------------------------------------
forwards           = df[df['position'] == 'Forward']
young_players      = df[df['age'] < 23]
high_rated         = df[df['player_rating'] > 8]
multi_condition    = df[(df['goals'] > 3) & (df['position'] == 'Forward')]
top_value_players  = df[df['market_value_eur'] > df['market_value_eur'].quantile(0.90)]

# loc / iloc examples
print("\nSelect specific columns with loc:\n",
      df.loc[:, ['player_name', 'team', 'goals', 'assists']].head())
print("\nSelect rows 0-4, columns 0-4 with iloc:\n", df.iloc[0:5, 0:5])

# query() style filtering
print("\nQuery example:\n",
      df.query("age < 25 and goals >= 2")[['player_name', 'age', 'goals']].head())

# -------------------------------------------------------------
# STEP 7 : SORTING
# -------------------------------------------------------------
top_scorers   = df.sort_values(by='goals', ascending=False).head(10)
top_rated     = df.sort_values(by='player_rating', ascending=False).head(10)
sorted_multi  = df.sort_values(by=['team', 'goals'], ascending=[True, False])

# -------------------------------------------------------------
# STEP 8 : GROUPBY / AGGREGATION
# -------------------------------------------------------------
goals_by_team = df.groupby('team')['goals'].sum().sort_values(ascending=False)
print("\nTotal goals by team (top 10):\n", goals_by_team.head(10))

avg_rating_by_position = df.groupby('position')['player_rating'].mean()
print("\nAvg rating by position:\n", avg_rating_by_position)

agg_summary = df.groupby('team').agg(
    total_goals=('goals', 'sum'),
    avg_rating=('player_rating', 'mean'),
    total_assists=('assists', 'sum'),
    players_count=('player_id', 'count')
).reset_index().sort_values('total_goals', ascending=False)
print("\nTeam-wise aggregated summary:\n", agg_summary.head(10))

# Multiple groupby keys
team_position_stats = df.groupby(['team', 'position'])['goals'].sum()

# -------------------------------------------------------------
# STEP 9 : PIVOT TABLE / CROSSTAB
# -------------------------------------------------------------
pivot_goals = pd.pivot_table(df, values='goals', index='team',
                              columns='position', aggfunc='sum', fill_value=0)
print("\nPivot table (goals: team x position):\n", pivot_goals.head())

cross_tab = pd.crosstab(df['position'], df['preferred_foot'])
print("\nCrosstab (position x preferred foot):\n", cross_tab)

# -------------------------------------------------------------
# STEP 10 : CORRELATION & COVARIANCE
# -------------------------------------------------------------
corr_matrix = df[num_cols].corr()
print("\nCorrelation matrix (subset):\n",
      corr_matrix[['goals', 'assists', 'player_rating']].head())

cov_matrix = df[num_cols].cov()

# Correlation of one column with all others
print("\nCorrelation of player_rating with other stats:\n",
      corr_matrix['player_rating'].sort_values(ascending=False).head(10))

# -------------------------------------------------------------
# STEP 11 : OUTLIER DETECTION (IQR METHOD)
# -------------------------------------------------------------
def detect_outliers_iqr(column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return df[(df[column] < lower) | (df[column] > upper)]

outliers_market_value = detect_outliers_iqr('market_value_eur')
print("\nOutliers in market_value_eur:", outliers_market_value.shape[0])

# Z-score method (NumPy)
z_scores = (df['player_rating'] - df['player_rating'].mean()) / df['player_rating'].std()
outliers_zscore = df[np.abs(z_scores) > 3]
print("Outliers via z-score in player_rating:", outliers_zscore.shape[0])

# -------------------------------------------------------------
# STEP 12 : FEATURE ENGINEERING / TRANSFORMATIONS
# -------------------------------------------------------------
df['goal_contribution']   = df['goals'] + df['assists']
df['pass_efficiency']     = df['successful_passes'] / df['total_passes'].replace(0, np.nan)
df['bmi']                 = df['weight_kg'] / ((df['height_cm'] / 100) ** 2)
df['age_group']           = pd.cut(df['age'], bins=[0, 20, 25, 30, 35, 50],
                                     labels=['<20', '20-25', '25-30', '30-35', '35+'])
df['is_high_value']       = np.where(df['market_value_eur'] > df['market_value_eur'].median(), 1, 0)

# Normalization / Standardization (NumPy)
df['rating_normalized'] = (df['player_rating'] - df['player_rating'].min()) / \
                           (df['player_rating'].max() - df['player_rating'].min())
df['rating_standardized'] = (df['player_rating'] - df['player_rating'].mean()) / df['player_rating'].std()

# -------------------------------------------------------------
# STEP 13 : APPLY / MAP / LAMBDA
# -------------------------------------------------------------
df['performance_tag'] = df['player_rating'].apply(
    lambda x: 'Excellent' if x >= 8 else ('Good' if x >= 6 else 'Average'))

foot_map = {'Left': 0, 'Right': 1, 'Both': 2}
df['preferred_foot_encoded'] = df['preferred_foot'].map(foot_map)

# -------------------------------------------------------------
# STEP 14 : NUMPY ARRAY OPERATIONS (direct array-level EDA)
# -------------------------------------------------------------
goals_array = df['goals'].to_numpy()
print("\nNumPy array of goals (first 10):", goals_array[:10])
print("Cumulative sum of goals (first 10):", np.cumsum(goals_array)[:10])
print("Sorted unique goal values:", np.unique(goals_array))
print("Where goals > 5:", np.where(goals_array > 5)[0][:10])
print("Any player with red card > 0:", np.any(df['red_cards'].to_numpy() > 0))
print("All players played > 0 minutes:", np.all(df['minutes_played'].to_numpy() > 0))

# -------------------------------------------------------------
# STEP 15 : DATA TYPE CONVERSIONS
# -------------------------------------------------------------
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')
df['age'] = df['age'].astype(int)
df['team'] = df['team'].astype('category')

# -------------------------------------------------------------
# STEP 16 : RENAMING / DROPPING COLUMNS
# -------------------------------------------------------------
df_renamed = df.rename(columns={'goals': 'goals_scored', 'assists': 'assists_made'})
df_dropped_cols = df.drop(columns=['jersey_number'])

# -------------------------------------------------------------
# STEP 17 : EXPORT CLEANED DATA
# -------------------------------------------------------------
df.to_csv('fifa_cleaned_data.csv', index=False)
print("\nCleaned data exported successfully as 'fifa_cleaned_data.csv'")

# -------------------------------------------------------------
# STEP 18 : QUICK EDA SUMMARY REPORT
# -------------------------------------------------------------
with open('cleaning_report.txt', 'w') as f:
    f.write(f"Total rows: {df.shape[0]}\n")
    f.write(f"Total columns: {df.shape[1]}\n")
    f.write(f"Missing values total: {df.isnull().sum().sum()}\n")
    f.write(f"Duplicate rows: {df.duplicated().sum()}\n")
    f.write(f"Numeric columns: {len(num_cols)}\n")
    f.write(f"Top scoring team: {goals_by_team.index[0]} ({goals_by_team.iloc[0]} goals)\n")

print("\nEDA complete. Report saved to 'cleaning_report.txt'.")

Shape (rows, cols): (54600, 75)

Column names:
 ['player_id', 'player_name', 'age', 'nationality', 'team', 'jersey_number', 'position', 'height_cm', 'weight_kg', 'preferred_foot', 'club_name', 'market_value_eur', 'match_id', 'match_date', 'stadium', 'city', 'opponent_team', 'tournament_stage', 'match_result', 'goals_team', 'goals_opponent', 'minutes_played', 'goals', 'assists', 'shots', 'shots_on_target', 'expected_goals_xg', 'expected_assists_xa', 'key_passes', 'successful_passes', 'total_passes', 'pass_accuracy', 'dribbles_attempted', 'successful_dribbles', 'crosses', 'successful_crosses', 'tackles', 'interceptions', 'clearances', 'blocks', 'aerial_duels_won', 'aerial_duels_lost', 'recoveries', 'defensive_actions', 'fouls_committed', 'fouls_suffered', 'yellow_cards', 'red_cards', 'offsides', 'saves', 'save_percentage', 'punches', 'clean_sheet', 'goals_conceded', 'penalty_saves', 'distance_covered_km', 'sprint_distance_km', 'top_speed_kmh', 'accelerations', 'decelerations', 'stamina_s

In [ ]:
# =============================================================
# FIFA WORLD CUP 2026 - PLAYER PERFORMANCE : DATA VISUALIZATION EDA
# Covers almost every common Matplotlib + Seaborn chart used in EDA
# =============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# STEP 0 : LOAD DATA
# -------------------------------------------------------------
RAW_PATH = 'fifa_world_cup_2026_player_performance.csv'   # apna path yahan set karo
df = pd.read_csv(RAW_PATH)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

num_cols = df.select_dtypes(include=np.number).columns.tolist()

# -------------------------------------------------------------
# STEP 1 : UNIVARIATE - HISTOGRAM (distribution of a numeric column)
# -------------------------------------------------------------
plt.figure()
sns.histplot(df['age'], bins=20, kde=True, color='skyblue')
plt.title('Distribution of Player Age')
plt.xlabel('Age')
plt.ylabel('Count')
plt.savefig('01_hist_age.png')
plt.close()

plt.figure()
sns.histplot(df['player_rating'], bins=25, kde=True, color='orange')
plt.title('Distribution of Player Rating')
plt.savefig('02_hist_rating.png')
plt.close()

# -------------------------------------------------------------
# STEP 2 : BOXPLOT (spread + outliers)
# -------------------------------------------------------------
plt.figure()
sns.boxplot(x=df['market_value_eur'], color='lightgreen')
plt.title('Boxplot - Market Value (EUR)')
plt.savefig('03_box_market_value.png')
plt.close()

plt.figure()
sns.boxplot(x='position', y='player_rating', data=df, palette='Set2')
plt.title('Player Rating by Position')
plt.xticks(rotation=30)
plt.savefig('04_box_rating_by_position.png')
plt.close()

# -------------------------------------------------------------
# STEP 3 : VIOLIN PLOT (distribution shape + density)
# -------------------------------------------------------------
plt.figure()
sns.violinplot(x='preferred_foot', y='player_rating', data=df, palette='muted')
plt.title('Rating Distribution by Preferred Foot')
plt.savefig('05_violin_rating_foot.png')
plt.close()

# -------------------------------------------------------------
# STEP 4 : COUNTPLOT (categorical frequency)
# -------------------------------------------------------------
plt.figure()
sns.countplot(y='position', data=df, order=df['position'].value_counts().index, palette='viridis')
plt.title('Number of Players by Position')
plt.savefig('06_count_position.png')
plt.close()

plt.figure()
top_nats = df['nationality'].value_counts().head(10)
sns.barplot(x=top_nats.values, y=top_nats.index, palette='crest')
plt.title('Top 10 Nationalities by Player Count')
plt.xlabel('Count')
plt.savefig('07_bar_top_nationalities.png')
plt.close()

# -------------------------------------------------------------
# STEP 5 : BAR CHART (aggregated values)
# -------------------------------------------------------------
goals_by_team = df.groupby('team')['goals'].sum().sort_values(ascending=False).head(10)
plt.figure()
sns.barplot(x=goals_by_team.values, y=goals_by_team.index, palette='rocket')
plt.title('Top 10 Teams by Total Goals')
plt.xlabel('Total Goals')
plt.savefig('08_bar_goals_by_team.png')
plt.close()

# -------------------------------------------------------------
# STEP 6 : SCATTER PLOT (relationship between 2 numeric vars)
# -------------------------------------------------------------
plt.figure()
sns.scatterplot(x='expected_goals_xg', y='goals', data=df, hue='position', alpha=0.6)
plt.title('Expected Goals (xG) vs Actual Goals')
plt.savefig('09_scatter_xg_vs_goals.png')
plt.close()

plt.figure()
sns.scatterplot(x='distance_covered_km', y='player_rating', data=df, alpha=0.5, color='purple')
plt.title('Distance Covered vs Player Rating')
plt.savefig('10_scatter_distance_rating.png')
plt.close()

# -------------------------------------------------------------
# STEP 7 : PAIRPLOT (multiple relationships at once)
# -------------------------------------------------------------
subset_cols = ['goals', 'assists', 'player_rating', 'market_value_eur']
sns.pairplot(df[subset_cols].sample(1000, random_state=42))   # sample for speed on large data
plt.savefig('11_pairplot_subset.png')
plt.close()

# -------------------------------------------------------------
# STEP 8 : CORRELATION HEATMAP
# -------------------------------------------------------------
plt.figure(figsize=(14, 10))
key_metrics = ['goals', 'assists', 'shots', 'shots_on_target', 'expected_goals_xg',
               'pass_accuracy', 'tackles', 'interceptions', 'player_rating',
               'market_value_eur', 'age']
corr = df[key_metrics].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap - Key Performance Metrics')
plt.savefig('12_heatmap_correlation.png')
plt.close()

# -------------------------------------------------------------
# STEP 9 : LINE PLOT (trend over time)
# -------------------------------------------------------------
df['match_date'] = pd.to_datetime(df['match_date'], errors='coerce')
goals_over_time = df.groupby(df['match_date'].dt.date)['goals'].sum()
plt.figure()
goals_over_time.plot(kind='line', marker='o', color='teal')
plt.title('Total Goals Scored Over Match Dates')
plt.xlabel('Match Date')
plt.ylabel('Goals')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('13_line_goals_over_time.png')
plt.close()

# -------------------------------------------------------------
# STEP 10 : PIE CHART (proportion / share)
# -------------------------------------------------------------
plt.figure()
foot_counts = df['preferred_foot'].value_counts()
plt.pie(foot_counts, labels=foot_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('pastel'))
plt.title('Preferred Foot Distribution')
plt.savefig('14_pie_preferred_foot.png')
plt.close()

# -------------------------------------------------------------
# STEP 11 : KDE PLOT (smooth density comparison)
# -------------------------------------------------------------
plt.figure()
for pos in df['position'].unique():
    sns.kdeplot(df[df['position'] == pos]['player_rating'], label=pos, fill=True, alpha=0.3)
plt.title('Player Rating Density by Position')
plt.legend()
plt.savefig('15_kde_rating_by_position.png')
plt.close()

# -------------------------------------------------------------
# STEP 12 : FACETGRID (multiple subplots by category)
# -------------------------------------------------------------
g = sns.FacetGrid(df, col='preferred_foot', height=5)
g.map(sns.histplot, 'player_rating', bins=20, color='steelblue')
g.set_titles('{col_name}')
g.savefig('16_facetgrid_rating_by_foot.png')
plt.close()

# -------------------------------------------------------------
# STEP 13 : JOINTPLOT (scatter + distribution combined)
# -------------------------------------------------------------
sns.jointplot(x='top_speed_kmh', y='player_rating', data=df, kind='hex', color='darkorange')
plt.savefig('17_jointplot_speed_rating.png')
plt.close()

# -------------------------------------------------------------
# STEP 14 : SUBPLOTS (multiple charts in one figure - dashboard style)
# -------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.histplot(df['age'], bins=20, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Age Distribution')

sns.boxplot(x='position', y='goals', data=df, ax=axes[0, 1], palette='Set3')
axes[0, 1].set_title('Goals by Position')
axes[0, 1].tick_params(axis='x', rotation=30)

sns.scatterplot(x='pass_accuracy', y='player_rating', data=df, ax=axes[1, 0], alpha=0.5)
axes[1, 0].set_title('Pass Accuracy vs Rating')

top5_teams = df.groupby('team')['player_rating'].mean().sort_values(ascending=False).head(5)
sns.barplot(x=top5_teams.values, y=top5_teams.index, ax=axes[1, 1], palette='mako')
axes[1, 1].set_title('Top 5 Teams by Avg Rating')

plt.tight_layout()
plt.savefig('18_dashboard_subplots.png')
plt.close()

print("All visualizations generated and saved as PNG files in the working directory.")

/tmp/ipykernel_1486/4123267154.py:49: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(x='position', y='player_rating', data=df, palette='Set2')
/tmp/ipykernel_1486/4123267154.py:59: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x='preferred_foot', y='player_rating', data=df, palette='muted')
/tmp/ipykernel_1486/4123267154.py:68: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(y='position', data=df, order=df['position'].value_counts().index, palette='viridis')
/tmp/ipykernel_1486/4123267154.py:75: FutureWarning: 

Passing `palette` without a

All visualizations generated and saved as PNG files in the working directory.


In [ ]:
from google.colab import files
files.download('fifa_cleaned_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>